##STEP 1: Mount Google Drive in Colab

In [1]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##STEP 2: Set Dataset Path

In [2]:
# Step 2: Define dataset path
dataset_path = "/content/drive/MyDrive/Pattern Recognition/ORL"

##STEP 3: Import Required Libraries

In [3]:
# Step 3: Import libraries
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE

##STEP 4: Load ORL Dataset

In [4]:
# Step 4: Load dataset

def load_orl_dataset(path):
    images = []
    labels = []

    label = 0
    for person in sorted(os.listdir(path)):
        person_path = os.path.join(path, person)

        if os.path.isdir(person_path):
            for image_name in os.listdir(person_path):
                img_path = os.path.join(person_path, image_name)

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (92, 112))
                img = img.flatten()

                images.append(img)
                labels.append(label)

            label += 1

    return np.array(images), np.array(labels)

X, y = load_orl_dataset(dataset_path)

print("Dataset shape:", X.shape)
print("Labels shape:", y.shape)

Dataset shape: (400, 10304)
Labels shape: (400,)


##STEP 5: Normalize Data

In [5]:
# Step 5: Normalize
X = X / 255.0

##STEP 6: Train-Test Split

In [7]:
# Step 6: Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (280, 10304)
Testing shape: (120, 10304)


##STEP 7: Apply PCA

In [8]:
# Step 7: PCA (reduce to 100 components)

pca = PCA(n_components=100)

X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("After PCA:")
print("Training shape:", X_train_pca.shape)
print("Testing shape:", X_test_pca.shape)

print("Explained variance ratio (first 5):")
print(pca.explained_variance_ratio_[:5])

After PCA:
Training shape: (280, 100)
Testing shape: (120, 100)
Explained variance ratio (first 5):
[0.17860144 0.12788855 0.06816555 0.05708432 0.05338324]


##STEP 8: k-NN Classification (PCA Features)

In [9]:
# Step 8: Apply k-NN

knn = KNeighborsClassifier(n_neighbors=3)

knn.fit(X_train_pca, y_train)

y_pred_pca = knn.predict(X_test_pca)

accuracy_pca = accuracy_score(y_test, y_pred_pca)

print("Accuracy using PCA + kNN:", accuracy_pca * 100, "%")

Accuracy using PCA + kNN: 94.16666666666667 %


##STEP 9: Filter Method (ANOVA SelectKBest) + kNN
###This selects the best K pixel features using ANOVA.

In [10]:
## STEP 9: Feature Selection – Filter Method (ANOVA)

from sklearn.feature_selection import SelectKBest, f_classif

# Select top 500 important features
selector_kbest = SelectKBest(score_func=f_classif, k=100)

X_train_kbest = selector_kbest.fit_transform(X_train, y_train)
X_test_kbest = selector_kbest.transform(X_test)

print("After SelectKBest:")
print("Training shape:", X_train_kbest.shape)
print("Testing shape:", X_test_kbest.shape)

After SelectKBest:
Training shape: (280, 100)
Testing shape: (120, 100)


In [11]:
# Apply kNN on filtered features
knn_kbest = KNeighborsClassifier(n_neighbors=3)

knn_kbest.fit(X_train_kbest, y_train)

y_pred_kbest = knn_kbest.predict(X_test_kbest)

accuracy_kbest = accuracy_score(y_test, y_pred_kbest)

print("Accuracy using SelectKBest + kNN:", accuracy_kbest * 100, "%")

Accuracy using SelectKBest + kNN: 58.333333333333336 %


##STEP 10: Wrapper Method

###Wrapper methods select features based on classifier performance.

In [12]:
from sklearn.feature_selection import SequentialFeatureSelector

# Reduce features first to 1000 (optional but faster)
selector = SelectKBest(score_func=f_classif, k=100)

X_train_reduced = selector.fit_transform(X_train, y_train)
X_test_reduced = selector.transform(X_test)

print("After initial filter:", X_train_reduced.shape)

After initial filter: (280, 100)


###10.1 : Apply Forward Selection + KNN

In [14]:
knn = KNeighborsClassifier(n_neighbors=3)

forward_selector = SequentialFeatureSelector(
    knn,
    n_features_to_select=50,
    direction='forward',
    cv=3,
    n_jobs=-1
)

X_train_forward = forward_selector.fit_transform(X_train_reduced, y_train)
X_test_forward = forward_selector.transform(X_test_reduced)

print("After Forward Selection:", X_train_forward.shape)

After Forward Selection: (280, 50)


In [15]:
knn.fit(X_train_forward, y_train)

y_pred_forward = knn.predict(X_test_forward)

accuracy_forward = accuracy_score(y_test, y_pred_forward)

print("Forward Wrapper + kNN Accuracy:", accuracy_forward * 100, "%")

Forward Wrapper + kNN Accuracy: 60.83333333333333 %


###10.2:Applying Backward Method + KNN

In [16]:
backward_selector = SequentialFeatureSelector(
    knn,
    n_features_to_select=50,
    direction='backward',
    cv=3,
    n_jobs=-1
)

X_train_backward = backward_selector.fit_transform(X_train_reduced, y_train)
X_test_backward = backward_selector.transform(X_test_reduced)

print("After Backward Selection:", X_train_backward.shape)

After Backward Selection: (280, 50)


In [17]:
knn.fit(X_train_backward, y_train)

y_pred_backward = knn.predict(X_test_backward)

accuracy_backward = accuracy_score(y_test, y_pred_backward)

print("Backward Wrapper + kNN Accuracy:", accuracy_backward * 100, "%")

Backward Wrapper + kNN Accuracy: 55.00000000000001 %


##Step 11: Result Comparison

In [18]:
print("\n---- Final Comparison ----")

print("PCA + kNN Accuracy:", accuracy_pca * 100, "%")
print("SelectKBest + kNN Accuracy:", accuracy_kbest * 100, "%")
print("Forward Wrapper + kNN Accuracy:", accuracy_forward * 100, "%")
print("Backward Wrapper + kNN Accuracy:", accuracy_backward * 100, "%")


---- Final Comparison ----
PCA + kNN Accuracy: 94.16666666666667 %
SelectKBest + kNN Accuracy: 58.333333333333336 %
Forward Wrapper + kNN Accuracy: 60.83333333333333 %
Backward Wrapper + kNN Accuracy: 55.00000000000001 %
